In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn import metrics
import statistics
import plotly.express as px

import warnings
warnings.filterwarnings("ignore")

pd.set_option('display.max_columns', None)

train_df = pd.read_csv("data/lc_trainingset.csv")
test_df = pd.read_csv("data/lc_testset.csv")

#Clean train_df set

def change_loan_status(loan_status):
    if loan_status in ['Fully Paid', 'Current']:
        return 0
    else:
        return 1

train_df['loan_status'] = train_df['loan_status'].apply(change_loan_status)

#remove 'month' string in term column and convert it to int

def term_to_float(term_string):
    if term_string == ' 36 months':
        return 36
    else:
        return 60

train_df['term']=train_df['term'].apply(term_to_float)

def grade_to_num(grade):
    grade_to_num = {'A':0, 'B':1, 'C':2, 'D':3, 'E':4, 'F':5, 'G':6}
    return grade_to_num[grade]

train_df['grade']=train_df['grade'].apply(grade_to_num)

#perform label encoding on sub_grade column

def subgrade_to_num(subgrade):
    subgrade_to_num = {'A1':0, 'A2':1, 'A3':2, 'A4':3, 'A5':4,
                      'B1':5, 'B2':6, 'B3':7, 'B4':8, 'B5':9,
                      'C1':10, 'C2':11, 'C3':12, 'C4':13, 'C5':14,
                      'D1':15, 'D2':16, 'D3':17, 'D4':18, 'D5':19,
                      'E1':20, 'E2':21, 'E3':22, 'E4':23, 'E5':24,
                      'F1':25, 'F2':26, 'F3':27, 'F4':28, 'F5':29,
                      'G1':30, 'G2':31, 'G3':32, 'G4':33, 'G5':34}
    return subgrade_to_num[subgrade]

train_df['sub_grade']=train_df['sub_grade'].apply(subgrade_to_num)

#perform label encoding on emp_length column, fill missing value with median value

def emplength_to_num(emp_length):
    emplength_to_num = {'< 1 year':0, '1 year' :1, '2 years':2, '3 years':3, '4 years':4, 
                        '5 years':5, '6 years':6, '7 years':7,'8 years': 8, '9 years':9,'10+ years':10}
    return emplength_to_num[emp_length]

train_df['emp_length']=train_df['emp_length'].apply(lambda x: emplength_to_num(x) if pd.notnull(x) else x)
train_df['emp_length']=train_df['emp_length'].fillna(train_df['emp_length'].median())

#find the time duration (in months) between 'issue_d' and 'earliest_cr_line' and store the value in a new variable

import numpy as np
import datetime

issue_date = pd.to_datetime(train_df['issue_d'])
earliest_credit_date = pd.to_datetime(train_df['earliest_cr_line'])
train_df['Credit_line_duration_mths'] = ((issue_date - earliest_credit_date)/np.timedelta64(1, 'M')).astype(int)


#fill missing value in 'revol_util' column with median value 

train_df['revol_util'] = train_df['revol_util'].fillna(train_df['revol_util'].median())

#fill missing value in 'pub_rec_bankruptcies' column with median value
 
train_df['pub_rec_bankruptcies'] = train_df['pub_rec_bankruptcies'].fillna(train_df['pub_rec_bankruptcies'].median())


#extract zipcode from 'address' column

train_df['zip_code'] = train_df['address'].apply(lambda x: x[-5:])
train_df['zip_code']= train_df['zip_code'].apply(str)

#drop the following features:
#drop id 
#drop 'address' as zipcode have been extracted
#drop 'grade' as similar info is captured under 'subgrade'
#drop emp_title (much data cleaning required, relationship to target could be represented by annual_inc feature)
#drop title (much data cleaning required, relationship to target could be represented by purpose feature)
#drop 'issue_d' and 'earliest_cr_line' as the key data(length of credit history before loan) is stored in new variable


train_df_clean = train_df.drop(columns = ['id',
                                'address',
                                'grade',
                                'emp_title',
                                'title',
                                'issue_d',
                                'earliest_cr_line'])


#remove rows with outliers in any numerical features 

numerical_features = ['loan_amnt', 'int_rate', 'installment', 
       'annual_inc', 'dti', 'open_acc', 'pub_rec', 'revol_bal', 'revol_util',
       'total_acc', 'mort_acc','pub_rec_bankruptcies', 'Credit_line_duration_mths']

outlier_indexes = []

for feature in numerical_features:
    upper_lim = train_df_clean[feature].quantile(.99) #quantitle can be adjusted
    lower_lim = train_df_clean[feature].quantile(.01)
    outliers = train_df_clean.index[(train_df_clean[feature] > upper_lim) | (train_df_clean[feature] < lower_lim)].tolist()
    print('No. of outliers in ' + feature + ' =' + str(len(outliers)))
    outlier_indexes += outliers #this contains duplicate indexes

row_index_to_drop = list(set(outlier_indexes)) #removes duplicate indexes

print(len(row_index_to_drop))
train_df_clean = train_df_clean.drop(row_index_to_drop)
train_df_clean = train_df_clean.reset_index(drop=True)
train_df_clean.shape

#scale the numerical features after removing outliers

from sklearn.preprocessing import StandardScaler

numerical_features = ['loan_amnt', 'int_rate', 'installment', 
       'annual_inc', 'dti', 'open_acc', 'pub_rec', 'revol_bal', 'revol_util',
       'total_acc', 'mort_acc','pub_rec_bankruptcies', 'Credit_line_duration_mths']

X_num = train_df_clean[numerical_features]
sc = StandardScaler()
sc.fit(X_num)
X_num_std = sc.transform(X_num)

X_num_std = pd.DataFrame(X_num_std, columns=numerical_features)

for feature in numerical_features:
    train_df_clean[feature] = X_num_std[feature]

train_df_clean.describe().T

from catboost import CatBoostClassifier

features = ['loan_amnt', 'term', 'int_rate', 'installment', 'sub_grade',
       'emp_length', 'home_ownership', 'annual_inc', 'verification_status',
       'purpose', 'dti', 'open_acc', 'pub_rec', 'revol_bal', 'revol_util',
       'total_acc', 'initial_list_status', 'application_type', 'mort_acc',
       'pub_rec_bankruptcies', 'Credit_line_duration_mths', 'zip_code']

#indicate 'home_ownership', 'verification_status', 'purpose', 'initial_list_status', 'application_type', 'zip_code' as cat_features

X = train_df_clean[features]
y = train_df_clean['loan_status']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.30, random_state=3)

Catboost_model = CatBoostClassifier(task_type = 'CPU', iterations = 5000, depth = 6, random_seed=3)

Catboost_model.fit(X_train, y_train, cat_features=([6, 8, 9, 16, 17,21]))

probability = Catboost_model.predict_proba(X_test)
y_pred_probability = probability[:,1]
fpr,tpr,threshold = metrics.roc_curve(y_test, y_pred_probability)
AUC = metrics.auc(fpr, tpr)

print(AUC)


#clean test_df set

#remove 'month' string in term column and convert it to int

def term_to_float(term_string):
    if term_string == ' 36 months':
        return 36
    else:
        return 60

test_df['term']=test_df['term'].apply(term_to_float)

def grade_to_num(grade):
    grade_to_num = {'A':0, 'B':1, 'C':2, 'D':3, 'E':4, 'F':5, 'G':6}
    return grade_to_num[grade]

test_df['grade']=test_df['grade'].apply(grade_to_num)

#perform label encoding on sub_grade column

def subgrade_to_num(subgrade):
    subgrade_to_num = {'A1':0, 'A2':1, 'A3':2, 'A4':3, 'A5':4,
                      'B1':5, 'B2':6, 'B3':7, 'B4':8, 'B5':9,
                      'C1':10, 'C2':11, 'C3':12, 'C4':13, 'C5':14,
                      'D1':15, 'D2':16, 'D3':17, 'D4':18, 'D5':19,
                      'E1':20, 'E2':21, 'E3':22, 'E4':23, 'E5':24,
                      'F1':25, 'F2':26, 'F3':27, 'F4':28, 'F5':29,
                      'G1':30, 'G2':31, 'G3':32, 'G4':33, 'G5':34}
    return subgrade_to_num[subgrade]

test_df['sub_grade']=test_df['sub_grade'].apply(subgrade_to_num)

#perform label encoding on emp_length column, fill missing value with median value

def emplength_to_num(emp_length):
    emplength_to_num = {'< 1 year':0, '1 year' :1, '2 years':2, '3 years':3, '4 years':4, 
                        '5 years':5, '6 years':6, '7 years':7,'8 years': 8, '9 years':9,'10+ years':10}
    return emplength_to_num[emp_length]

test_df['emp_length']=test_df['emp_length'].apply(lambda x: emplength_to_num(x) if pd.notnull(x) else x)


#find the time duration (in months) between 'issue_d' and 'earliest_cr_line' and store the value in a new variable

import numpy as np
import datetime

issue_date = pd.to_datetime(test_df['issue_d'], format='%b-%y')
earliest_credit_date = pd.to_datetime(test_df['earliest_cr_line'], format='%b-%y')
test_df['Credit_line_duration_mths'] = ((issue_date - earliest_credit_date)/np.timedelta64(1, 'M')).astype(int)


#extract zipcode from 'address' column

test_df['zip_code'] = test_df['address'].apply(lambda x: x[-5:])
test_df['zip_code']= test_df['zip_code'].apply(str)

#drop the following features:
#drop id 
#drop 'address' as zipcode & state have been extracted
#drop 'grade' as similar info is captured under 'subgrade'
#drop emp_title (much data cleaning required, relationship to target could be represented by annual_inc feature)
#drop title (much data cleaning required, relationship to target could be represented by purpose feature)
#drop 'issue_d' and 'earliest_cr_line' as the key data(length of credit history before loan) is stored in new variable


test_df_clean = test_df.drop(columns = ['id',
                                'address',
                                'grade',
                                'emp_title',
                                'title',
                                'issue_d',
                                'earliest_cr_line'])


#scale the numerical features 

from sklearn.preprocessing import StandardScaler

numerical_features = ['loan_amnt', 'int_rate', 'installment', 
       'annual_inc', 'dti', 'open_acc', 'pub_rec', 'revol_bal', 'revol_util',
       'total_acc', 'mort_acc','pub_rec_bankruptcies', 'Credit_line_duration_mths']

X_num = test_df_clean[numerical_features]
sc = StandardScaler()
sc.fit(X_num)
X_num_std = sc.transform(X_num)

X_num_std = pd.DataFrame(X_num_std, columns=numerical_features)

for feature in numerical_features:
    test_df_clean[feature] = X_num_std[feature]

test_df_clean.describe().T

from catboost import CatBoostClassifier

features = ['loan_amnt', 'term', 'int_rate', 'installment', 'sub_grade',
       'emp_length', 'home_ownership', 'annual_inc', 'verification_status',
       'purpose', 'dti', 'open_acc', 'pub_rec', 'revol_bal', 'revol_util',
       'total_acc', 'initial_list_status', 'application_type', 'mort_acc',
       'pub_rec_bankruptcies', 'Credit_line_duration_mths', 'zip_code']

#indicate 'home_ownership', 'verification_status', 'purpose', 'initial_list_status','application_type', 'zip_code' as cat_features


X_train = train_df_clean[features]
y_train = train_df_clean['loan_status']
X_test = test_df_clean[features]

#fit model on all train_set data
Catboost_model.fit(X_train, y_train, cat_features=([6, 8, 9, 16, 17,21])) 
probabilities = Catboost_model.predict_proba(X_test)
kaggle_preds = probabilities[:,1]
len(kaggle_preds)

output_dataframe = pd.DataFrame({
    'Id': list(range(len(kaggle_preds))),
    'Predicted': kaggle_preds})

output_dataframe.to_csv('my_predictions.csv', index=False)  


No. of outliers in loan_amnt =3257
No. of outliers in int_rate =6127
No. of outliers in installment =6314
No. of outliers in annual_inc =5724
No. of outliers in dti =6316
No. of outliers in open_acc =4267
No. of outliers in pub_rec =2008
No. of outliers in revol_bal =6336
No. of outliers in revol_util =6256
No. of outliers in total_acc =5994
No. of outliers in mort_acc =1669
No. of outliers in pub_rec_bankruptcies =1880
No. of outliers in Credit_line_duration_mths =6209
47364
Learning rate set to 0.022063
0:	learn: 0.6473176	total: 125ms	remaining: 10m 24s
1:	learn: 0.6094038	total: 175ms	remaining: 7m 18s
2:	learn: 0.5742856	total: 225ms	remaining: 6m 14s
3:	learn: 0.5445520	total: 279ms	remaining: 5m 48s
4:	learn: 0.5169306	total: 332ms	remaining: 5m 31s
5:	learn: 0.4903373	total: 392ms	remaining: 5m 26s
6:	learn: 0.4673476	total: 454ms	remaining: 5m 23s
7:	learn: 0.4484126	total: 507ms	remaining: 5m 16s
8:	learn: 0.4318870	total: 555ms	remaining: 5m 7s
9:	learn: 0.4160520	total: 610

In [2]:
print(AUC)

0.9089380743246923
